# Resumen documento

A lo largo de este documento simula la solución al **Minimal Maximal Matching** de varios grafos de ejemplo

In [1]:
from bloqade.analog import start, cast, load, save
import os
import matplotlib.pyplot as plt
from bokeh.io import output_notebook
from bloqade.analog.atom_arrangement import ListOfLocations, Square
from bloqade.analog import piecewise_linear, cast
import numpy as np

output_notebook()

if not os.path.isdir("data"):
    os.mkdir("data")

c:\Users\javip\Documents\Códigos_TFG\Codigo_tfg\.venv\Lib\site-packages\bloqade\analog\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


Loading BokehJS ...

# Condiciones para la simulación

## Constantes $A,\space B, \space C$

$$
\begin{cases}
    A>(\Delta-2)B \\
    B>C
\end{cases}
$$

Extra para imponer bien el bloqueo de Rydberg: $$A\gg B$$

## Parámetros Hardware: $\Delta_{ij}, \space d_{ij}$

### Términos lineales: $\Delta_{ij}$

$$
\begin{equation}
    \Delta_i = - B\sum_{e\in E}(\deg(u)+\deg(v))x_e + B|E| + C\sum_e x_e
\end{equation}
$$

### Términos no lineales: $d_{ij}$

Tomamos $A \gg B \implies W_{ij} \simeq A$. Pero si no tendríamos $W_{12}=A+2B \quad ; \quad W_{13}=B$.

$$
\begin{cases}
    d_{ij} = \left( \frac{C_6}{A} \right) ^{1/6} \\
    A = \frac{C_6}{d_{ij}}
\end{cases}
$$

Además, para imponer bloqueo: $d_{ij}<R_B \implies \Omega<A = \frac{C_6}{d_{ij}}$

In [56]:
import itertools
from IPython.display import display, HTML # para mostrar mensajes de error en rojo y negrita

# ══════════════════════════════════════════════════════════════
# PASO 1: Definición del grafo
# ══════════════════════════════════════════════════════════════
# Aristas del grafo original (pares de vértices)
edges = {
    'e0': ('A', 'B'),
    'e1': ('B', 'C'),
    'e2': ('C', 'D'),
    'e3': ('C', 'E'),
}
edge_list = list(edges.keys())  # ['e0', 'e1', 'e2', 'e3']

# Grado de cada vértice
from collections import defaultdict
degree = defaultdict(int)
for u, v in edges.values():
    degree[u] += 1
    degree[v] += 1

delta_max = max(degree.values())
print("── PASO 1: Análisis del grafo ──")
#print(f"  Aristas : {edges}")
#print(f"  Grados  : {dict(degree)}")
print(f"  Δ (grado máximo) = {delta_max}")



# ══════════════════════════════════════════════════════════════
# PASO 2: Elección de A, B, C respetando la jerarquía
# ══════════════════════════════════════════════════════════════
# Condición: A > (Δ-2)B  y  B > C > 0
# Con Δ=3 → A > B > C
A = 200.0
B = 3   # Mayor B → menos aristas coloreadas
C_cost = 0.5   # Mayor C_cost → menos aristas coloreadas
delta_max = 3

print(f"\n── Paso 2: Valores A, B, C ──")
print(f"  A={A}, B={B}, C={C_cost}")
if A > (delta_max - 2) * B and B > C_cost > 0:
    print("  Condiciones cumplidas:")
    print(f"  A > (Δ-2)B → {A} > {(delta_max-2)*B:.2f} ✓")
    print(f"  B > C → {B} > {C_cost} ✓")
else:
    display(HTML("<span style='color: #FF0000; font-weight: bold; font-size: 1.2em;'>Condiciones A,B,C no cumplidas:</span>"))
    print(f"  A > (Δ-2)B → {A} > {(delta_max-2)*B:.2f} ")
    print(f"  B > C → {B} > {C_cost} ")






# ══════════════════════════════════════════════════════════════
# PASO 3: Coeficientes lineales → Detuning local Δᵢ
# ══════════════════════════════════════════════════════════════
#   → Δᵢ = B·(deg(u) + deg(v)) + B|E| − C 

print(f"\n── PASO 3: Detuning local ──")
delta_local = {}
for e, (u, v) in edges.items():
    coef_HB = B * (degree[u] + degree[v])
    coef_constante = B * len(edges) # Este término es constante y realmetne no afecta, pero lo incluyo para comprobarlo
    coef_HC = C_cost
    delta_local[e] = coef_HB + coef_constante - coef_HC
    print(f"  {e}=({u},{v}): deg({u})={degree[u]}, deg({v})={degree[v]}"
          f"  →  Δ_{e} = {B}·{degree[u]+degree[v]} + {B}·{len(edges)} − {C_cost} = {delta_local[e]:.3f}")




# ══════════════════════════════════════════════════════════════
# PASO 4: Distancias físicas a partir de A
# ══════════════════════════════════════════════════════════════
C6 = 2 * np.pi * 862690  # [rad/µs · µm^6]

# Radio de bloqueo de Rydberg
R_B = (C6 / omega_max) ** (1/6)
print(f"\n── PASO 4: Distancias físicas ──")
print(f"  R_B = (C6/Ω)^(1/6) = {R_B:.2f} µm")

# Distancia entre átomos con interacción (W_ij ≈ A)
d = (C6 / A) ** (1/6)
print(f"  d = (C6/A)^(1/6) = {d:.2f} µm")

# Verificación de bloqueo: d < R_B ↔ Ω < A
if d < R_B:
    print(f"  Condición de bloqueo: d={d:.2f} < R_B={R_B:.2f} ✓  (A={A} > Ω={omega_max})")
else:
    display(HTML("<span style='color: #FF0000; font-weight: bold;'>Bloqueo NO garantizado: d > R_B . Aumenta A o reduce Ω </span>"))
    print(f"  d={d:.2f} > R_B={R_B:.2f}  →  aumenta A o reduce Ω")


# ══════════════════════════════════════════════════════════════
# PASO 4.2: Cálculo de distancias exactas considerando términos de B
# ══════════════════════════════════════════════════════════════
print(f"\n── 4.2 NOTA EXTRA: Distancias exactas (W_ij = A + M·B)  ──")
d_required_exact = {}
for e1, e2 in itertools.combinations(edge_list, 2):
    u1, v1 = edges[e1]
    u2, v2 = edges[e2]
    shared = {u1, v1} & {u2, v2}

    V_HA = A if shared else 0
    V_HB = B * len(shared)
    if not shared:
        for w in {u1, v1}:
            for x in {u2, v2}:
                if any(({w, x} <= {u, v}) for u, v in edges.values()):
                    V_HB += B
                    break

    V_ij_exact = V_HA + V_HB
    d_ij_exact = (C6 / V_ij_exact) ** (1/6) if V_ij_exact > 0 else np.inf
    d_required_exact[(e1, e2)] = d_ij_exact

    bloqueo = "✓" if d_ij_exact < R_B else "⚠"
    print(f"  ({e1},{e2}): W={V_HA}+{V_HB:.1f}={V_ij_exact:.2f}  →  d_exact={d_ij_exact:.2f} µm  "
          f"vs  d_aprox={d_ij:.2f} µm  {bloqueo}")



# ══════════════════════════════════════════════════════════════
# PASO 5: Geometría ajustada con las distancias
# ══════════════════════════════════════════════════════════════
print(f"\n── PASO 5: Geometría ajustada con las distancias ──")

d_required = {}
coordenadas = [
    (-d, 0.0),
    (0.0, 0.0),
    (d * np.cos(np.pi/4),  d * np.sin(np.pi/4)),
    (d * np.cos(-np.pi/4), d * np.sin(-np.pi/4)),
]




# ══════════════════════════════════════════════════════════════
# PASO 6: Waveforms y secuencia Bloqade
# ══════════════════════════════════════════════════════════════
omega_max  = 3.3
delta_end  = 8.0
sweep_time = 15.0
durations  = [0.8, sweep_time, 0.8]
rabi_amplitude_values = [0.0, omega_max, omega_max, 0.0]
rabi_detuning_values  = [-delta_end, -delta_end, delta_end, delta_end]

# escala_i = Δᵢ / delta_end  (para que el detuning efectivo final sea Δᵢ)
escalas = [delta_local[e] / delta_end for e in edge_list]

print(f"\n── PASO 7: Escalas de detuning ──")
for e, esc in zip(edge_list, escalas):
    print(f"  {e}: Δ={delta_local[e]:.3f}  →  escala = {esc:.4f}  →  Δ efectivo = {esc*delta_end:.3f}")

geometria = ListOfLocations(coordenadas)
prog = (
    geometria
    .rydberg.rabi.amplitude.uniform.piecewise_linear(durations, rabi_amplitude_values)
    .detuning.scale(escalas).piecewise_linear(durations, rabi_detuning_values)
)
#geometria.show()

emu_prog = prog.bloqade.python().run(shots=30000)

emu_prog.report().show()

── PASO 1: Análisis del grafo ──
  Δ (grado máximo) = 3

── Paso 2: Valores A, B, C ──
  A=200.0, B=3, C=0.5
  Condiciones cumplidas:
  A > (Δ-2)B → 200.0 > 3.00 ✓
  B > C → 3 > 0.5 ✓

── PASO 3: Detuning local ──
  e0=(A,B): deg(A)=1, deg(B)=2  →  Δ_e0 = 3·3 + 3·4 − 0.5 = 20.500
  e1=(B,C): deg(B)=2, deg(C)=3  →  Δ_e1 = 3·5 + 3·4 − 0.5 = 26.500
  e2=(C,D): deg(C)=3, deg(D)=1  →  Δ_e2 = 3·4 + 3·4 − 0.5 = 23.500
  e3=(C,E): deg(C)=3, deg(E)=1  →  Δ_e3 = 3·4 + 3·4 − 0.5 = 23.500

── PASO 4: Distancias físicas ──
  R_B = (C6/Ω)^(1/6) = 10.86 µm
  d = (C6/A)^(1/6) = 5.48 µm
  Condición de bloqueo: d=5.48 < R_B=10.86 ✓  (A=200.0 > Ω=3.3)

── 4.2 NOTA EXTRA: Distancias exactas (W_ij = A + M·B)  ──
  (e0,e1): W=200.0+3.0=203.00  →  d_exact=5.47 µm  vs  d_aprox=6.15 µm  ✓
  (e0,e2): W=0+3.0=3.00  →  d_exact=11.04 µm  vs  d_aprox=6.15 µm  ⚠
  (e0,e3): W=0+3.0=3.00  →  d_exact=11.04 µm  vs  d_aprox=6.15 µm  ⚠
  (e1,e2): W=200.0+3.0=203.00  →  d_exact=5.47 µm  vs  d_aprox=6.15 µm  ✓
  (e1,e3): W=